In [2]:
#imports
import pandas as pd
import scipy.stats as stats
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import accuracy_score

In [21]:
#load all dfs
#dummy_2018 = pd.read_csv('dummy_2018.csv')
#dummy_2019 = pd.read_csv('dummy_2019.csv')
#dummy_2020 = pd.read_csv('dummy_2020.csv')
#dummy_2021 = pd.read_csv('dummy_2021.csv')
dummy_2022 = pd.read_csv('dummy_2022.csv')
dummy_2022

,activity_year,action_taken,loan_amount,loan_to_value_ratio,loan_term,property_value,total_units,income,applicant_race_observed,co-applicant_race_observed,...,applicant_age_above_62_Yes,submission_of_application_Submitted directly to your institution,initially_payable_to_institution_Not applicable,initially_payable_to_institution_Not initially payable to your institution,aus-1_Guaranteed Underwriting System (GUS),aus-1_Internal Proprietary System,aus-1_Loan Prospector (LP) or Loan Product Advisor,aus-1_Not applicable,aus-1_Other,aus-1_Technology Open to Approved Lenders (TOTAL) Scorecard
0,2022,1,135000.0,90.000,360.0,155000.0,1,33.0,2,4,...,0,1,0,0,0,0,0,1,0,0
1,2022,0,185000.0,90.000,360.0,205000.0,3,52.0,2,4,...,0,1,0,0,0,0,0,1,0,0
2,2022,1,255000.0,86.897,360.0,295000.0,1,72.0,1,4,...,0,1,0,0,0,0,0,1,0,0
3,2022,1,165000.0,72.727,360.0,225000.0,1,38.0,1,4,...,0,1,0,0,0,0,0,1,0,0
4,2022,1,265000.0,85.000,360.0,315000.0,1,69.0,2,2,...,0,1,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83403,2022,1,295000.0,80.000,360.0,365000.0,1,150.0,2,2,...,0,1,0,0,0,0,0,0,0,0
83404,2022,1,205000.0,95.000,360.0,225000.0,1,86.0,2,4,...,0,1,0,0,0,0,0,0,0,0
83405,2022,1,205000.0,90.000,360.0,235000.0,1,129.0,2,4,...,0,1,0,0,0,0,1,0,0,0
83406,2022,1,525000.0,85.000,360.0,615000.0,1,120.0,2,4,...,0,1,0,0,0,0,1,0,0,0


In [15]:
corr_matrix_2022 = dummy_2022.corr()
corr_matrix_2022 = corr_matrix_2022.dropna(axis='columns', how='all')
corr_matrix_2022 = corr_matrix_2022.dropna(axis='index', how='all')
corr_matrix_2022

,action_taken,loan_amount,loan_to_value_ratio,property_value,total_units,income,applicant_race_observed,co-applicant_race_observed,tract_population,tract_minority_population_percent,...,applicant_age_above_62_Yes,submission_of_application_Submitted directly to your institution,initially_payable_to_institution_Not applicable,initially_payable_to_institution_Not initially payable to your institution,aus-1_Guaranteed Underwriting System (GUS),aus-1_Internal Proprietary System,aus-1_Loan Prospector (LP) or Loan Product Advisor,aus-1_Not applicable,aus-1_Other,aus-1_Technology Open to Approved Lenders (TOTAL) Scorecard
action_taken,1.000000,0.031069,-0.101234,0.047256,-0.125615,0.133374,0.029952,-0.054256,0.024200,-0.162128,...,-0.009303,0.041573,-0.361300,0.020079,-0.001753,-0.121790,0.093524,-0.164058,-0.045431,-0.098005
loan_amount,0.031069,1.000000,-0.071456,0.953420,0.091817,0.634256,-0.043289,-0.235324,0.135373,-0.041063,...,-0.090279,-0.087824,-0.007633,0.054265,-0.084117,-0.004233,0.054280,-0.037165,-0.013146,-0.077748
loan_to_value_ratio,-0.101234,-0.071456,1.000000,-0.312620,0.035608,-0.171661,0.029072,0.081961,-0.071850,0.176325,...,-0.139478,-0.004291,0.041770,-0.035399,0.067275,-0.010367,-0.134136,-0.029861,-0.010612,0.234136
property_value,0.047256,0.953420,-0.312620,1.000000,0.074891,0.648703,-0.047805,-0.242674,0.141455,-0.085876,...,-0.053439,-0.075461,-0.019182,0.059540,-0.089513,0.015458,0.080865,-0.008282,0.005738,-0.136067
total_units,-0.125615,0.091817,0.035608,0.074891,1.000000,-0.009618,-0.008561,0.051178,-0.066276,0.236920,...,-0.015824,-0.033527,0.038149,-0.003733,-0.013108,0.000874,-0.036204,0.016994,0.036903,0.096102
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
aus-1_Internal Proprietary System,-0.121790,-0.004233,-0.010367,0.015458,0.000874,-0.002031,-0.024303,0.004368,-0.002306,0.013955,...,0.020964,0.019719,-0.010511,-0.020019,-0.006486,1.000000,-0.046521,-0.016487,-0.004967,-0.030124
aus-1_Loan Prospector (LP) or Loan Product Advisor,0.093524,0.054280,-0.134136,0.080865,-0.036204,0.055844,-0.046506,-0.007504,0.030678,-0.068621,...,-0.003914,-0.060100,-0.033718,0.078626,-0.046217,-0.046521,1.000000,-0.117485,-0.035394,-0.214662
aus-1_Not applicable,-0.164058,-0.037165,-0.029861,-0.008282,0.016994,-0.019261,-0.032297,0.000671,-0.011925,0.030036,...,0.009800,0.029757,0.016037,-0.002314,-0.016379,-0.016487,-0.117485,1.000000,-0.012543,-0.076076
aus-1_Other,-0.045431,-0.013146,-0.010612,0.005738,0.036903,-0.004595,-0.013660,-0.007559,-0.009666,0.035395,...,0.020382,0.020653,-0.007997,-0.012715,-0.004934,-0.004967,-0.035394,-0.012543,1.000000,-0.022919


In [17]:
def filter_by_index_start(df, lst):
    mask = df.index.to_series().apply(lambda x: any(x.startswith(string) for string in lst))
    filtered_df = df[mask]
    return filtered_df

In [22]:
def filter_by_index_columns(df, lst):
    # Filter rows based on index starting with any prefix in lst
    mask = df.index.to_series().apply(lambda x: any(x.startswith(prefix) for prefix in lst))
    filtered_df = df[mask]
    
    # Remove columns that start with any prefix found in lst
    filtered_df = filtered_df.loc[:, ~filtered_df.columns.to_series().apply(lambda col: any(col.startswith(prefix) for prefix in lst))]
    
    return filtered_df

In [25]:
lst = ['derived_ethnicity', 'derived_race', 'derived_sex', 'applicant_ethnicity', 'co-applicant_ethnicity']
filtered_matrix = filter_by_index_columns(corr_matrix_2022, lst)
filtered_matrix

,action_taken,loan_amount,loan_to_value_ratio,property_value,total_units,income,applicant_race_observed,co-applicant_race_observed,tract_population,tract_minority_population_percent,...,applicant_age_above_62_Yes,submission_of_application_Submitted directly to your institution,initially_payable_to_institution_Not applicable,initially_payable_to_institution_Not initially payable to your institution,aus-1_Guaranteed Underwriting System (GUS),aus-1_Internal Proprietary System,aus-1_Loan Prospector (LP) or Loan Product Advisor,aus-1_Not applicable,aus-1_Other,aus-1_Technology Open to Approved Lenders (TOTAL) Scorecard
derived_ethnicity_Free Form Text Only,-0.008226,-0.002652,0.001749,-0.002355,-0.001959,-0.005953,-0.001952,0.002034,-0.000781,0.007035,...,0.004320,0.005274,-0.001560,-0.002972,-0.000963,0.023936,-0.006907,0.002655,-0.000737,0.001635
derived_ethnicity_Hispanic or Latino,-0.059845,-0.036728,0.127996,-0.070114,0.073284,-0.157600,-0.018084,0.038151,0.019263,0.265047,...,-0.062742,-0.021523,0.015109,-0.032489,-0.029551,-0.001010,-0.057487,0.041824,0.016409,0.052852
derived_ethnicity_Joint,0.014685,0.054990,0.007681,0.048994,-0.006268,0.083217,0.004461,-0.188828,0.004379,-0.010681,...,-0.020986,0.012505,-0.006403,-0.000792,-0.007722,-0.001022,0.002363,-0.010939,0.005647,-0.005143
derived_ethnicity_Not Hispanic or Latino,0.078295,0.011839,-0.115404,0.043076,-0.069128,0.097080,-0.000667,0.023171,-0.013936,-0.231254,...,0.047929,-0.015733,-0.029091,0.053404,0.028754,0.005976,0.058233,-0.028024,-0.010128,-0.063969
derived_race_American Indian or Alaska Native,-0.011640,-0.023969,0.015945,-0.026021,0.005081,-0.032022,-0.001526,0.027162,0.001246,0.020269,...,-0.006053,0.012503,0.003233,-0.011193,-0.003377,0.005242,-0.005669,0.008074,-0.001416,0.009795
derived_race_Asian,-0.005290,0.135914,-0.091075,0.153409,-0.017712,0.075530,-0.027175,0.030154,0.058858,0.016723,...,-0.033287,-0.115462,-0.002756,0.022269,-0.020775,0.007412,0.039367,-0.011429,-0.004610,-0.063119
derived_race_Black or African American,-0.136992,-0.074388,0.194634,-0.117949,0.089149,-0.105389,-0.014106,0.129484,-0.085840,0.369937,...,0.053445,0.014613,0.052676,-0.039604,-0.026315,0.015843,-0.095996,0.014510,0.025661,0.198964
derived_race_Free Form Text Only,0.002159,-0.001517,0.004862,-0.002474,-0.000979,-0.002036,-0.000976,-0.000051,0.003060,0.000460,...,-0.001683,0.002637,-0.000780,-0.001486,-0.000481,0.024419,-0.003453,-0.001224,-0.000369,-0.002236
derived_race_Joint,0.011692,0.057788,-0.001824,0.054956,-0.013700,0.089994,0.003002,-0.184803,0.008716,-0.018508,...,-0.012757,0.020877,-0.005335,-0.007074,-0.000484,-0.000621,-0.004734,-0.001784,-0.005266,-0.009915
derived_race_Native Hawaiian or Other Pacific Islander,-0.016058,-0.005939,0.008492,-0.007342,0.008440,-0.007471,-0.003515,0.011052,-0.001413,0.008935,...,-0.001045,0.000678,-0.000914,0.002502,-0.002456,-0.002472,-0.009451,0.009776,0.004524,0.006568
